# Notebook 04 — Statistical Analysis

**Purpose:** Apply appropriate statistical tests for a three-class target to quantify associations between features and diabetes status, distinguishing statistical significance from practical significance.

**Inputs:** Cleaned dataset from Notebook 02

**Outputs:**
- Statistical results tables (`results/tables/04_*.csv`)
- Effect size reports


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import kruskal, chi2_contingency, spearmanr, f_oneway
import warnings
warnings.filterwarnings("ignore")

from src.config import *
from src.data_utils import (
    BINARY_FEATURES, ORDINAL_FEATURES, CONTINUOUS_FEATURES,
    ALL_ORIGINAL_FEATURES,
)

seed_everything()

df = pd.read_csv(os.path.join(DATA_PROCESSED, "df1_cleaned.csv"))
print(f"Dataset: {df.shape}")


Dataset: (253680, 22)


## 4.1 Descriptive Statistics by Class

In [2]:
desc_by_class = []
for cls in range(N_CLASSES):
    subset = df[df[TARGET_COL] == cls][ALL_ORIGINAL_FEATURES]
    desc = subset.describe().T[["mean", "std", "50%"]]
    desc.columns = [f"Mean_{CLASS_MAP[cls]}", f"Std_{CLASS_MAP[cls]}", f"Median_{CLASS_MAP[cls]}"]
    desc_by_class.append(desc)

desc_combined = pd.concat(desc_by_class, axis=1)
desc_combined.to_csv(os.path.join(RESULTS_TABLES, "04_descriptive_by_class.csv"))
print(desc_combined.round(3).to_string())


                      Mean_No Diabetes  Std_No Diabetes  Median_No Diabetes  Mean_Prediabetes  Std_Prediabetes  Median_Prediabetes  Mean_Diabetes  Std_Diabetes  Median_Diabetes
HighBP                           0.371            0.483                 0.0             0.629            0.483                 1.0          0.753         0.431              1.0
HighChol                         0.379            0.485                 0.0             0.621            0.485                 1.0          0.670         0.470              1.0
CholCheck                        0.957            0.203                 1.0             0.987            0.115                 1.0          0.993         0.082              1.0
Smoker                           0.430            0.495                 0.0             0.493            0.500                 0.0          0.518         0.500              1.0
Stroke                           0.032            0.175                 0.0             0.057            0.232     

## 4.2 Kruskal-Wallis Test (Numerical Features)

For a three-class target with ordinal/continuous features, the Kruskal-Wallis H-test is more appropriate than ANOVA when normality assumptions may not hold (which is typical for survey data with skewed distributions).


In [3]:
kw_results = []
groups = [df[df[TARGET_COL]==c] for c in range(N_CLASSES)]

for feat in CONTINUOUS_FEATURES + ORDINAL_FEATURES:
    samples = [g[feat].values for g in groups]
    stat, pval = kruskal(*samples)
    
    # Effect size: eta-squared approximation for Kruskal-Wallis
    n = len(df)
    k = N_CLASSES
    eta_sq = (stat - k + 1) / (n - k)
    
    kw_results.append({
        "Feature": feat,
        "H_statistic": stat,
        "p_value": pval,
        "Eta_squared": max(0, eta_sq),
        "Significant_0.05": "Yes" if pval < 0.05 else "No",
        "Effect_Size": "Large" if eta_sq > 0.14 else ("Medium" if eta_sq > 0.06 else "Small"),
    })

kw_df = pd.DataFrame(kw_results).sort_values("H_statistic", ascending=False)
kw_df.to_csv(os.path.join(RESULTS_TABLES, "04_kruskal_wallis.csv"), index=False)
print(kw_df.to_string(index=False))

print("\n⚠ Note on statistical vs practical significance:")
print("  With n=253,680, almost every test will be statistically significant.")
print("  We focus on EFFECT SIZE (Eta²) to identify practically important features.")


  Feature  H_statistic       p_value  Eta_squared Significant_0.05 Effect_Size
  GenHlth 22480.925024  0.000000e+00     0.088612              Yes      Medium
      BMI 14130.749997  0.000000e+00     0.055696              Yes       Small
      Age  8811.762696  0.000000e+00     0.034728              Yes       Small
   Income  7558.899340  0.000000e+00     0.029789              Yes       Small
 PhysHlth  6661.878029  0.000000e+00     0.026253              Yes       Small
Education  4083.037135  0.000000e+00     0.016088              Yes       Small
 MentHlth   528.910573 1.407759e-115     0.002077              Yes       Small

⚠ Note on statistical vs practical significance:
  With n=253,680, almost every test will be statistically significant.
  We focus on EFFECT SIZE (Eta²) to identify practically important features.


## 4.3 Chi-Square Tests (Categorical/Binary Features)

In [4]:
chi2_results = []

for feat in BINARY_FEATURES:
    ct = pd.crosstab(df[feat], df[TARGET_COL])
    chi2, pval, dof, expected = chi2_contingency(ct)
    
    # Cramér's V effect size
    n = ct.values.sum()
    k = min(ct.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * k)) if k > 0 else 0
    
    chi2_results.append({
        "Feature": feat,
        "Chi2": chi2,
        "p_value": pval,
        "dof": dof,
        "Cramers_V": cramers_v,
        "Significant_0.05": "Yes" if pval < 0.05 else "No",
        "Effect_Size": "Large" if cramers_v > 0.5 else ("Medium" if cramers_v > 0.3 else ("Small" if cramers_v > 0.1 else "Negligible")),
    })

chi2_df = pd.DataFrame(chi2_results).sort_values("Chi2", ascending=False)
chi2_df.to_csv(os.path.join(RESULTS_TABLES, "04_chi_square.csv"), index=False)
print(chi2_df.to_string(index=False))


             Feature         Chi2       p_value  dof  Cramers_V Significant_0.05 Effect_Size
              HighBP 18794.644052  0.000000e+00    2   0.272191              Yes       Small
            DiffWalk 12776.941889  0.000000e+00    2   0.224425              Yes       Small
            HighChol 11258.920399  0.000000e+00    2   0.210671              Yes       Small
HeartDiseaseorAttack  8244.889107  0.000000e+00    2   0.180281              Yes       Small
        PhysActivity  3789.301463  0.000000e+00    2   0.122218              Yes       Small
              Stroke  2916.751980  0.000000e+00    2   0.107228              Yes       Small
           CholCheck  1173.749358 1.329124e-255    2   0.068021              Yes  Negligible
              Smoker  1010.511751 3.716732e-220    2   0.063114              Yes  Negligible
             Veggies   893.841905 8.029646e-195    2   0.059359              Yes  Negligible
   HvyAlcoholConsump   850.324048 2.261930e-185    2   0.057896       

## 4.4 Spearman Correlation with Target

In [5]:
spearman_results = []
for feat in ALL_ORIGINAL_FEATURES:
    rho, pval = spearmanr(df[feat], df[TARGET_COL])
    spearman_results.append({
        "Feature": feat,
        "Spearman_rho": rho,
        "abs_rho": abs(rho),
        "p_value": pval,
        "Direction": "Positive" if rho > 0 else "Negative",
    })

sp_df = pd.DataFrame(spearman_results).sort_values("abs_rho", ascending=False)
sp_df.to_csv(os.path.join(RESULTS_TABLES, "04_spearman_correlations.csv"), index=False)
print(sp_df[["Feature", "Spearman_rho", "p_value", "Direction"]].to_string(index=False))


             Feature  Spearman_rho       p_value Direction
             GenHlth      0.297138  0.000000e+00  Positive
              HighBP      0.271668  0.000000e+00  Positive
                 BMI      0.235887  0.000000e+00  Positive
            DiffWalk      0.223567  0.000000e+00  Positive
            HighChol      0.210668  0.000000e+00  Positive
                 Age      0.186357  0.000000e+00  Positive
HeartDiseaseorAttack      0.178564  0.000000e+00  Positive
              Income     -0.172611  0.000000e+00  Negative
            PhysHlth      0.161718  0.000000e+00  Positive
           Education     -0.126862  0.000000e+00  Negative
        PhysActivity     -0.121988  0.000000e+00  Negative
              Stroke      0.105887  0.000000e+00  Positive
           CholCheck      0.068018 8.431614e-258  Positive
              Smoker      0.063040 1.121521e-221  Positive
             Veggies     -0.059353 1.062420e-196  Negative
   HvyAlcoholConsump     -0.057244 4.328898e-183  Negati

## 4.5 Multiple Testing Correction

In [6]:
from statsmodels.stats.multitest import multipletests

# Combine all p-values for correction
all_pvals = list(kw_df["p_value"].values) + list(chi2_df["p_value"].values)
all_names = list(kw_df["Feature"].values) + list(chi2_df["Feature"].values)
all_tests = (["Kruskal-Wallis"] * len(kw_df)) + (["Chi-Square"] * len(chi2_df))

reject, pvals_corrected, _, _ = multipletests(all_pvals, method="fdr_bh")

correction_df = pd.DataFrame({
    "Feature": all_names,
    "Test": all_tests,
    "Original_p": all_pvals,
    "Corrected_p_BH": pvals_corrected,
    "Significant_after_correction": reject,
})
correction_df.to_csv(os.path.join(RESULTS_TABLES, "04_multiple_testing_correction.csv"), index=False)
print(correction_df.to_string(index=False))
print(f"\n{reject.sum()} / {len(reject)} tests remain significant after BH correction.")


             Feature           Test    Original_p  Corrected_p_BH  Significant_after_correction
             GenHlth Kruskal-Wallis  0.000000e+00    0.000000e+00                          True
                 BMI Kruskal-Wallis  0.000000e+00    0.000000e+00                          True
                 Age Kruskal-Wallis  0.000000e+00    0.000000e+00                          True
              Income Kruskal-Wallis  0.000000e+00    0.000000e+00                          True
            PhysHlth Kruskal-Wallis  0.000000e+00    0.000000e+00                          True
           Education Kruskal-Wallis  0.000000e+00    0.000000e+00                          True
            MentHlth Kruskal-Wallis 1.407759e-115   1.738996e-115                          True
              HighBP     Chi-Square  0.000000e+00    0.000000e+00                          True
            DiffWalk     Chi-Square  0.000000e+00    0.000000e+00                          True
            HighChol     Chi-Square  0.0

## 4.6 Practical Significance Summary

In [7]:
# Combine effect sizes from all tests
effect_summary = []
for _, row in kw_df.iterrows():
    effect_summary.append({
        "Feature": row["Feature"], "Test": "Kruskal-Wallis",
        "Effect_Metric": "Eta²", "Effect_Value": row["Eta_squared"],
        "Effect_Category": row["Effect_Size"],
    })
for _, row in chi2_df.iterrows():
    effect_summary.append({
        "Feature": row["Feature"], "Test": "Chi-Square",
        "Effect_Metric": "Cramér's V", "Effect_Value": row["Cramers_V"],
        "Effect_Category": row["Effect_Size"],
    })

effect_df = pd.DataFrame(effect_summary).sort_values("Effect_Value", ascending=False)
effect_df.to_csv(os.path.join(RESULTS_TABLES, "04_effect_sizes.csv"), index=False)
print("Features ranked by effect size:")
print(effect_df.to_string(index=False))

print("\n" + "="*60)
print("KEY INSIGHT: Statistical vs Practical Significance")
print("="*60)
print("With n=253,680, virtually ALL features are statistically significant.")
print("However, effect sizes vary considerably:")
print("- Features with MEDIUM/LARGE effects are genuinely important discriminators.")
print("- Features with SMALL/NEGLIGIBLE effects, despite tiny p-values,")
print("  have limited practical value for distinguishing diabetes classes.")


Features ranked by effect size:
             Feature           Test Effect_Metric  Effect_Value Effect_Category
              HighBP     Chi-Square    Cramér's V      0.272191           Small
            DiffWalk     Chi-Square    Cramér's V      0.224425           Small
            HighChol     Chi-Square    Cramér's V      0.210671           Small
HeartDiseaseorAttack     Chi-Square    Cramér's V      0.180281           Small
        PhysActivity     Chi-Square    Cramér's V      0.122218           Small
              Stroke     Chi-Square    Cramér's V      0.107228           Small
             GenHlth Kruskal-Wallis          Eta²      0.088612          Medium
           CholCheck     Chi-Square    Cramér's V      0.068021      Negligible
              Smoker     Chi-Square    Cramér's V      0.063114      Negligible
             Veggies     Chi-Square    Cramér's V      0.059359      Negligible
   HvyAlcoholConsump     Chi-Square    Cramér's V      0.057896      Negligible
        

## 4.7 Conclusion

**Statistical analysis findings:**

1. **All features** are statistically significant (p < 0.001) — expected with n=253,680.
2. **GenHlth, Age, BMI, HighBP, DiffWalk** have the largest effect sizes, confirming EDA findings.
3. **Multiple testing correction** (Benjamini-Hochberg FDR) does not change any conclusions.
4. **Spearman correlations** show GenHlth, HighBP, Age, and BMI have the strongest monotonic relationships with the target.
5. **Practical significance** must guide feature selection — small but statistically significant effects may not contribute meaningfully to the model.

**Next:** Notebook 05 — Feature Engineering
